# F1 AI Strategy Simulator

## Step 2 — Feature Engineering

In this notebook we transform the raw telemetry dataset into machine-learning features.

Feature engineering extracts meaningful signals from race data.

These features will later be used for:

- Lap time prediction
- Overtake probability modeling
- Pit stop prediction
- Race simulation

The final dataset will be saved as:

data/feature_dataset.csv

In [1]:
import pandas as pd
import numpy as np

## Load Telemetry Dataset

We load the cleaned telemetry dataset created in Step 1.

In [26]:
df = pd.read_csv("../data/race_dataset.csv")

print("Rows:", len(df))
df.head()

# Remove Pit Laps (noisy)
df = df[df["IsPitLap"] == 0]


Rows: 159009


# Encode Driver
Machine learning models cannot use text features directly.

In [3]:
df["DriverEncoded"] = df["Driver"].astype("category").cat.codes

# Encode Team
Team performance influences lap pace.

In [4]:
df["TeamEncoded"] = df["Team"].astype("category").cat.codes

# Encode Track
Tracks differ significantly in pace and overtaking difficulty.

In [5]:
df["TrackEncoded"] = df["Track"].astype("category").cat.codes

## Encode Tire Compound

Machine learning models cannot use text values directly.

We convert tire compounds into numeric values.

In [6]:
compound_map = {
    "SOFT":0,
    "MEDIUM":1,
    "HARD":2,
    "INTERMEDIATE":3,
    "WET":4
}

df["CompoundEncoded"] = df["Compound"].map(compound_map)

## Tire Age Feature

Tyre life already exists in the dataset.

We create a clean feature called `tyre_age`.

In [7]:
df["tyre_age"] = df["TyreLife"]

# Sort Dataset

Sorting ensures rolling calculations work correctly.

In [8]:
df = df.sort_values(["Season","Track","Driver","LapNumber"])

# Pace Trend Feature

Drivers’ pace evolves due to:

- tire degradation
- fuel burn
- track evolution

We compute the average of the **previous 3 laps**.

Using `.shift(1)` avoids data leakage.

In [9]:
df["pace_trend"] = (
    df.groupby(["Season","Track","Driver"])["LapTimeSeconds"]
    .shift(1)
    .rolling(3)
    .mean()
)

# Laps Since Pit

Pit stops reset tyre performance.

In [10]:
df["pit_event"] = df["IsPitLap"]

df["pit_group"] = (
    df.groupby(["Season","Track","Driver"])["pit_event"]
    .cumsum()
)

df["laps_since_pit"] = (
    df.groupby(["Season","Track","Driver","pit_group"])
    .cumcount()
)

In [11]:
df["stint_number"] = (
    df.groupby(["Season","Track","Driver"])["IsPitLap"]
    .cumsum()
)

In [12]:
track_pace = (
    df.groupby(["Season","Track"])["LapTimeSeconds"]
    .mean()
    .rename("track_pace_baseline")
)

df = df.merge(track_pace, on=["Season","Track"])

# Fuel Load Feature

Cars start heavy with fuel and get lighter each lap.

Approximate burn rate:

~1.7 kg per lap

In [13]:
race_length = df.groupby(["Season","Track"])["LapNumber"].transform("max")

df["fuel_load"] = (race_length - df["LapNumber"]) * 1.7

# Track Evolution Feature

Grip improves as rubber builds on the track.

In [14]:
df["track_evolution"] = df["LapNumber"] / race_length
df["lap_progress"] = df["LapNumber"] / df.groupby(
    ["Season","Track"]
)["LapNumber"].transform("max")

# Compute Cumulative Race Time

Approximate the race clock by summing lap times.

In [15]:
df["race_time"] = (
    df.groupby(["Season","Track","Driver"])["LapTimeSeconds"]
    .cumsum()
)

# Compute Gaps Between Cars

Use cumulative race time to estimate realistic track gaps.

In [16]:
df = df.sort_values(["Season","Track","LapNumber","Position"])

df["gap_ahead"] = (
    df.groupby(["Season","Track","LapNumber"])["race_time"]
    .diff()
)

df["gap_behind"] = (
    df.groupby(["Season","Track","LapNumber"])["race_time"]
    .diff(-1)
)
df["traffic_pressure"] = df["gap_ahead"] - df["gap_behind"]

# DRS Availability

DRS becomes available when a driver is within ~1 second of the car ahead.

In [17]:
df["drs_available"] = (
    (df["gap_ahead"] > 0) & (df["gap_ahead"] < 1)
).astype(int)

# Driver Pace Baseline (Per Race)

Compute each driver's average pace **within the same race**.

In [18]:
driver_pace = (
    df.groupby(["Season","Track","Driver"])["LapTimeSeconds"]
    .mean()
    .rename("driver_pace_baseline")
)

df = df.merge(driver_pace, on=["Season","Track","Driver"])

# Relative Pace Feature

In [19]:
df["pace_relative"] = (
    df["LapTimeSeconds"] - df["driver_pace_baseline"]
)

# Remove Early-Lap NaNs

Rolling features produce NaNs for the first laps.

In [20]:

# Remove Early-Lap NaNs
df = df.dropna(subset=["pace_trend"])

# Construct Feature Dataset

In [21]:
features = df[
[
"Driver",          # keep driver name
"DriverEncoded",
"TeamEncoded",
"TrackEncoded",
"Season",
"RegulationEra",
"LapNumber",
"lap_progress",
"Position",
"CompoundEncoded",
"tyre_age",
"stint_number",
"pace_trend",
"laps_since_pit",
"fuel_load",
"track_evolution",
"AirTemp",
"TrackTemp",
"Humidity",
"WindSpeed",
"race_time",
"gap_ahead",
"gap_behind",
"traffic_pressure",
"drs_available",
"driver_pace_baseline",
"track_pace_baseline",
"pace_relative",
"LapTimeSeconds"
]
]

# Save Feature Dataset

In [22]:
features.to_csv("../data/feature_dataset.csv", index=False)

print("Feature dataset saved successfully.")
print("Rows:", len(features))

Feature dataset saved successfully.
Rows: 145913
